In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time
import csv
from bs4 import BeautifulSoup

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

driver.get("https://www.jumia.co.ke/flash-sales")
print(f'-----Scraping Jumia Flash Sales-----')
all_products = []

while True:
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    products = soup.select("article.prd")
    # print(products)

    for product in products:
        product_link = product.select_one('a.core')
        url = "https://www.jumia.co.ke" + product_link['href'] if product_link else None
        # print(url)
        if not url:
            continue
        
        driver.get(url)
        time.sleep(2)
        
        product_soup = BeautifulSoup(driver.page_source, 'html.parser')
        # print(product_soup)

        print('-----------------------------------------------------------')
        product_name = product_soup.select_one('h1')
        name = product_name.get_text(strip=True) if product_name else 'N/A'
        print(f'Product name: {name}')

        # New price
        new_price_tag = product_soup.select_one('span.-b.-ubpt.-tal.-fs24')
        new_price = new_price_tag.get_text(strip=True) if new_price_tag else 'N/A'
        print(f'New Price: {new_price}')

        # Old price
        old_price_tag = product_soup.select_one("span.-tal.-gy5.-lthr.-fs16")
        old_price = old_price_tag.get_text(strip=True) if old_price_tag else "N/A"
        print(f'Old price: {old_price}')

        discount_tag = product_soup.select_one("span.bdg._dsct._dyn")
        discount = discount_tag.get_text(strip=True) if discount_tag else "N/A"
        print(f'Discount: {discount}')

        brand_tag = product_soup.select_one("div.-pvxs a._more")
        brand = brand_tag.get_text(strip=True) if brand_tag else "N/A"
        print(f'Brand: {brand}')

        rating_tag = product_soup.select_one('div.stars')
        rating = rating_tag.text.strip() if rating_tag else 'N/A'
        print(f'Rating {rating}')

        items_left_tag = product_soup.select_one('span.-fsh0.-prs.-fs12')
        items_left = items_left_tag.text.strip() if items_left_tag else 'N/a'
        print(items_left)
 
        timer_tag = product_soup.select_one("time")
        time_left = timer_tag.text.strip() if timer_tag else "N/A"
        print(f'Time left: {time_left}')
        print('-----------------------------------------------------------')

        all_products.append({
            "Product Name": name,
            "New Price":    new_price,
            "Old Price":    old_price,
            "Discount (%)": discount,
            "Brand":        brand,
            "Items Left":   items_left,
            "Rating":       rating,
            "Time Left":    time_left,
            'URL': url
        })

        driver.back()
        time.sleep(2)

    try:
        next_btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label='Next Page']")
        next_btn.click()
    except Exception:
        print("\nEnd of pages.")
        break

driver.quit()

if all_products:
    fieldnames = [
        "Product Name", "New Price", "Old Price", "Discount (%)", "Brand", "Items Left", "Rating", "Time Left", "URL"
    ]
    with open('products.csv', 'w', newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_products)
    print(f"\nSaved {len(all_products)} products to `products.csv`")
else:
    print("\nNo products scraped.")



-----Scraping Jumia Flash Sales-----
-----------------------------------------------------------
Product name: MATIHO   Electric hair clippers, bald head shaver, razor, hairdressing electric trimmer, oil head carving, edging electric hair shears
New Price: KSh 302
Old price: KSh 460
Discount: 34%
Brand: MATIHO
Rating 3.7 out of 5
N/a
Time left: N/A
-----------------------------------------------------------
-----------------------------------------------------------
Product name: Kuhl KUHL K1 20,000mAh Power Bank – High-Capacity Portable Charger for Phones & USB Devices
New Price: KSh 690
Old price: KSh 1,340
Discount: 49%
Brand: Kuhl
Rating 3.7 out of 5
N/a
Time left: N/A
-----------------------------------------------------------
-----------------------------------------------------------
Product name: BLWOENS Men's Formal Shoes Tuxedo Dress Shoes Classic Oxfords Faux Patent Leather Shoe Lace-up Dress Shoes for Business Wedding Suit - Black
New Price: KSh 799
Old price: KSh 2,025
Dis